# H&M 거래·상품 이미지 표본 추출

비공개 Kaggle Notebook에서 **Add Input**으로 H&M Personalized Fashion Recommendations 대회를 연결한 뒤 실행한다. 대회 입력은 Kaggle에서 읽고, 로컬로 옮길 파일은 이용 조건을 확인한 후 `hm_sample.zip` 하나만 선택한다. 표본 ZIP과 해제한 파일은 로컬 프로젝트의 `data/raw/hm/`에 둔다.

기본 기간은 2020년 6~8월이며 원본이 이 기간을 포함하는지 실행 중 검사한다. 초깃값은 고객 최대 1,000명과 실제 상품 이미지 최대 200장이다. 채널 코드가 확인될 때까지 두 채널을 함께 사용한다. 구매일과 고객 ID, 원래 가격을 보존하며, RFM 기준일은 표본 추출 전 범위에서 결정한다.

In [ ]:
# 파일 개요: Kaggle 원본에서 기간·고객·이미지 조건을 검증한 재현 가능한 H&M 표본 ZIP을 만든다.
from pathlib import Path	# 표본 경로를 운영체제에 맞는 경로 객체로 다룬다.

INPUT_ROOT = Path("/kaggle/input")	# Kaggle에 연결한 입력 자료의 기준 경로다.
DATASET_DIR = None	# 입력 후보가 여러 개이면 사용할 폴더 경로를 직접 지정한다.
OUTPUT_DIR = Path("/kaggle/working")	# 완성된 표본 ZIP을 저장할 작업 경로다.

START_DATE = "2020-06-01"	# 포함할 첫 거래일을 설정한다.
END_DATE = "2020-09-01"	# 이 날짜는 표본 기간에 포함하지 않는다.
SALES_CHANNEL = None	# 채널 의미가 확인되기 전에는 두 채널을 모두 사용한다.
SEED = 42	# 고객·이미지 추출을 재현할 난수 시드다.
MAX_CUSTOMERS = 1_000	# 고객 표본의 최대 크기를 제한한다.
MAX_IMAGES = 200	# ZIP에 포함할 상품 이미지 수를 제한한다.
MIN_TRANSACTIONS = 1_000	# 미션에서 요구하는 최소 거래 행 수를 검증한다.
CHUNK_SIZE = 250_000	# 전체 거래 CSV를 나누어 읽어 메모리 사용량을 제한한다.

In [ ]:
import json	# 추출 설정과 건수 근거를 JSON으로 저장한다.
from collections import Counter	# 청크별 제외 사유와 유효 행 수를 누적한다.
from tempfile import TemporaryDirectory	# 완성 전 파일을 임시 폴더에 보관하고 종료 시 정리한다.
from zipfile import ZIP_DEFLATED, ZIP_STORED, ZipFile	# 표 데이터와 이미지를 서로 다른 압축 방식으로 묶는다.

import numpy as np	# 전역 난수 상태와 분리된 표본 추출과 수치 검증에 사용한다.
import pandas as pd	# CSV 청크 처리와 거래일 검증에 사용한다.


TRANSACTION_COLUMNS = ["t_dat", "customer_id", "article_id", "price", "sales_channel_id"]	# 입력에서 분석에 필요한 거래 열만 읽는다.
ARTICLE_COLUMNS = ["article_id", "prod_name", "product_type_name"]	# 상품 식별자와 텍스트·분류 정보를 선택한다.


def find_dataset(input_root):	# 파일 구성으로 유효한 입력 폴더를 판별한다.
    """H&M 원본 데이터 폴더를 찾는다.

    입력: input_root는 연결된 자료를 검색할 상위 경로다.
    출력: 거래 CSV·상품 CSV·이미지 폴더를 갖춘 유일한 경로다.
    부작용: 파일 목록을 읽으며 원본과 출력 파일을 변경하지 않는다.
    후보가 없거나 여러 개이면 ValueError를 발생시킨다.
    """
    candidates = [
        path.parent
        for pattern in ("transactions_train.csv", "*/transactions_train.csv", "*/*/transactions_train.csv")
        for path in input_root.glob(pattern)
        if (path.parent / "articles.csv").is_file() and (path.parent / "images").is_dir()
    ]
    if len(candidates) != 1:	# 후보가 없거나 여러 개이면 임의 선택을 하지 않는다.
        raise ValueError("Attach the H&M competition input or set DATASET_DIR to its folder.")
    return candidates[0]	# 검증된 단일 원본 폴더를 반환한다.


def transaction_chunks(path, start, end, sales_channel, chunk_size):	# 전체 CSV 대신 유효 거래를 청크 단위로 순회한다.
    """유효한 거래와 청크별 검증 근거를 순차적으로 제공한다.

    입력: CSV 경로, 시작일·종료일, 선택 채널과 청크 크기다.
    출력: 선택 거래·거래일·원본 최소일·최대일·감사 건수의 반복자다.
    부작용: CSV를 읽고 내부 청크 사본을 정규화하며 파일은 쓰지 않는다.
    """
    for rows in pd.read_csv(path, usecols=TRANSACTION_COLUMNS, dtype="string", chunksize=chunk_size):	# 문자열로 읽어 식별자의 선행 0을 보존한다.
        rows["customer_id"] = rows["customer_id"].str.strip().replace("", pd.NA)	# 빈 고객 키를 결측값으로 통일한다.
        rows["article_id"] = rows["article_id"].str.strip().replace("", pd.NA)	# 상품 키의 공백과 빈 문자열을 정규화한다.
        dates = pd.to_datetime(rows["t_dat"], format="%Y-%m-%d", errors="coerce")	# 해석할 수 없는 날짜는 결측으로 표시한다.
        amounts = pd.to_numeric(rows["price"], errors="coerce").to_numpy(dtype=float, na_value=np.nan)	# 숫자로 변환할 수 없는 금액을 유효성 검증에서 제외한다.
        customer_ok = rows["customer_id"].notna()
        date_ok = dates.notna()
        amount_ok = np.isfinite(amounts) & (amounts > 0)	# 양수이면서 유한한 거래 금액만 허용한다.
        valid = customer_ok & date_ok & amount_ok	# 고객·날짜·금액 조건을 모두 만족하는 행을 고른다.
        in_scope = dates.ge(start) & dates.lt(end)	# 시작일 포함·종료일 제외 조건을 적용한다.
        if sales_channel is not None:
            in_scope &= rows["sales_channel_id"].eq(str(sales_channel)).fillna(False)	# 채널이 지정된 경우에만 추가로 범위를 제한한다.
        audit = {	# 제외 사유가 겹치지 않도록 우선순위에 따라 건수를 기록한다.
            "source_rows": len(rows),
            "missing_customer_rows": int((~customer_ok).sum()),
            "invalid_date_rows": int((customer_ok & ~date_ok).sum()),
            "invalid_amount_rows": int((customer_ok & date_ok & ~amount_ok).sum()),
            "out_of_scope_rows": int((valid & ~in_scope).sum()),
        }
        selected = valid & in_scope	# 유효성 조건과 기간·채널 조건을 결합한다.
        yield rows.loc[selected, TRANSACTION_COLUMNS], dates.loc[selected], dates.min(), dates.max(), audit	# 선택 거래와 원본 범위 및 감사 건수를 함께 전달한다.


def choose(values, limit, seed):	# 입력 순서가 달라도 같은 표본을 재현한다.
    """입력 순서와 독립적인 재현 가능한 부분집합을 고른다.

    입력: 후보 값 모음, 선택 개수 상한과 난수 시드다.
    출력: 후보 수와 상한 중 작은 개수의 중복 없는 값 집합이다.
    부작용: 지역 난수 생성기를 사용하며 전역 난수 상태와 입력을 바꾸지 않는다.
    """
    ordered = sorted(values)	# 난수 추출 전에 후보 순서를 고정한다.
    indices = np.random.default_rng(seed).choice(len(ordered), size=min(limit, len(ordered)), replace=False)	# 중복 없이 지정 상한만큼 선택한다.
    return {ordered[index] for index in indices}	# 후속 포함 여부 검사에 사용할 집합을 반환한다.


def prepare_hm_sample(dataset_dir, output_dir, *, start, end, seed, max_customers,	# 고객 기준의 거래 표본과 이미지 부분집합을 만든다.
                      max_images, min_transactions, chunk_size, sales_channel=None):
    """거래·상품·이미지 표본과 추출 근거를 하나의 ZIP으로 만든다.

    입력: 원본·출력 경로, 기간·시드·표본 상한·최소 행 수·청크 크기·채널이다.
    출력: 완성 ZIP 경로와 표본 설정 및 검증 건수를 담은 사전이다.
    부작용: 출력 폴더와 임시 파일을 만들고 성공 시 hm_sample.zip을 교체한다.
    원본 자료는 읽기만 하며 임시 폴더는 함수 종료 시 정리한다.
    """
    start, end = pd.Timestamp(start), pd.Timestamp(end)	# 기간 비교를 위해 날짜 형식을 통일한다.
    if not (start.is_month_start and end.is_month_start and start + pd.DateOffset(months=3) <= end):	# 완전한 세 달 이상이라는 미션 조건을 확인한다.
        raise ValueError("Use month starts for an interval of at least three complete months.")
    if min(max_customers, max_images, min_transactions, chunk_size) < 1:	# 빈 표본과 잘못된 청크 크기를 미리 거부한다.
        raise ValueError("Sampling limits and chunk size must be positive.")

    transaction_path = dataset_dir / "transactions_train.csv"
    customers, covered_months = set(), set()
    counts = Counter()
    reference_date = source_start = source_end = None
    for rows, dates, raw_start, raw_end, audit in transaction_chunks(	# 첫 번째 순회에서 전체 후보 고객과 원본 기간을 확인한다.
        transaction_path, start, end, sales_channel, chunk_size
    ):
        counts.update(audit)	# 청크별 감사 건수를 하나의 기록으로 누적한다.
        if pd.notna(raw_start):
            source_start = raw_start if source_start is None else min(source_start, raw_start)
            source_end = raw_end if source_end is None else max(source_end, raw_end)
        if not rows.empty:
            customers.update(rows["customer_id"].unique())	# 기간·채널 조건을 통과한 고객을 후보 집합에 넣는다.
            covered_months.update(dates.dt.to_period("M").unique())
            reference_date = dates.max() if reference_date is None else max(reference_date, dates.max())	# 표본 추출 이전 유효 거래의 마지막 날짜를 RFM 기준일로 고정한다.
            counts["eligible_rows"] += len(rows)

    expected_months = set(pd.period_range(start, end - pd.Timedelta(days=1), freq="M"))	# 설정한 기간에 있어야 할 월 집합을 계산한다.
    if (source_start is None or source_start > start or source_end < end - pd.Timedelta(days=1)	# 원본 기간과 실제 관측 월이 설정 전체를 포함하는지 검증한다.
            or covered_months != expected_months):
        raise ValueError("The source and channel do not cover the configured complete months.")
    chosen_customers = choose(customers, max_customers, seed)	# 재현 가능한 고객 표본을 선택한다.
    population_size = len(customers)
    del customers	# 두 번째 순회 전에 전체 후보 집합 메모리를 해제한다.

    output_dir.mkdir(parents=True, exist_ok=True)	# 표본 산출물 경로를 준비한다.
    destination = output_dir / "hm_sample.zip"
    with TemporaryDirectory(dir=output_dir) as temp_dir:	# 검증을 마치기 전까지 기존 완성 ZIP을 교체하지 않는다.
        work = Path(temp_dir)
        sample_csv = work / "transactions_sample.csv"
        transaction_count = 0
        article_ids = set()
        for rows, _, _, _, _ in transaction_chunks(transaction_path, start, end, sales_channel, chunk_size):	# 두 번째 순회에서 선택 고객의 모든 유효 거래를 모은다.
            sample = rows.loc[rows["customer_id"].isin(chosen_customers)]
            if sample.empty:
                continue
            sample.to_csv(sample_csv, mode="a", header=transaction_count == 0, index=False)	# 청크를 이어 저장하되 열 이름은 처음에만 기록한다.
            transaction_count += len(sample)
            article_ids.update(sample["article_id"].dropna().unique())	# 거래 표본에 등장하는 상품 키만 수집한다.
        if transaction_count < min_transactions:	# 거래 수가 미션 최소 기준에 미달하면 중단한다.
            raise ValueError(f"Only {transaction_count} eligible sample rows. Increase MAX_CUSTOMERS or the period.")

        articles = pd.read_csv(dataset_dir / "articles.csv", usecols=ARTICLE_COLUMNS, dtype="string")	# 상품 표에서도 필요한 열만 문자열로 읽는다.
        articles = articles.loc[articles["article_id"].isin(article_ids)].copy()	# 거래 표본에 포함된 상품으로 메타데이터를 제한한다.
        if articles["article_id"].duplicated().any():	# 상품 키 중복으로 거래 조인이 늘어나는 문제를 차단한다.
            raise ValueError("Duplicate product IDs in the selected article metadata.")
        image_paths = {	# 숫자 상품 식별자만 원본 이미지 경로로 변환한다.
            article_id: dataset_dir / "images" / article_id[:3] / f"{article_id}.jpg"
            for article_id in articles["article_id"]
            if article_id.isascii() and article_id.isdecimal()
        }
        image_paths = {article_id: path for article_id, path in image_paths.items() if path.is_file()}	# 실제 존재하는 이미지 파일만 후보로 남긴다.
        if not image_paths:
            raise ValueError("No product images matched the selected transactions and metadata.")
        chosen_images = choose(image_paths, max_images, seed)	# 거래 표본과 별도로 이미지 표본의 크기를 제한한다.
        articles["source_image_available"] = articles["article_id"].isin(image_paths)	# 원본에 이미지가 없는 경우와 미표본 상태를 구분한다.
        articles["image_sampled"] = articles["article_id"].isin(chosen_images)	# 선택된 이미지의 상품 여부를 메타데이터에 기록한다.
        articles.to_csv(work / "articles_sample.csv", index=False)	# 후속 분석에서 사용할 상품 표를 저장한다.

        counts["sampled_rows"] = transaction_count
        counts["sample_excluded_rows"] = counts["eligible_rows"] - transaction_count	# 유효 거래 중 고객 표본에 포함되지 않은 행을 기록한다.
        excluded = sum(counts[name] for name in (	# 모든 제외 사유의 합계를 계산한다.
            "missing_customer_rows", "invalid_date_rows", "invalid_amount_rows",
            "out_of_scope_rows", "sample_excluded_rows",
        ))
        if counts["sample_excluded_rows"] < 0 or counts["source_rows"] != excluded + transaction_count:	# 원본 행 수가 제외 행과 표본 행으로 정확히 분해되는지 확인한다.
            raise ValueError("Transaction counts could not be reconciled.")
        manifest = {	# 설정·기준일·표본 규모와 감사 건수를 재현 근거로 묶는다.
            "source": "h-and-m-personalized-fashion-recommendations",
            "source_start": str(source_start.date()),
            "source_end": str(source_end.date()),
            "start_inclusive": str(start.date()),
            "end_exclusive": str(end.date()),
            "sales_channel": sales_channel,
            "seed": seed,
            "reference_date": str(reference_date.date()),
            "eligible_customers": population_size,
            "sampled_customers": len(chosen_customers),
            "sampled_product_ids": len(article_ids),
            "matched_products": len(articles),
            "matched_products_without_images": len(articles) - len(image_paths),
            "products_with_images_outside_image_sample": len(image_paths) - len(chosen_images),
            "sampled_images": len(chosen_images),
            "transactions": dict(counts),
        }
        (work / "sample_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")	# 추출 근거를 ZIP에 포함할 JSON 파일로 저장한다.
        archive_path = work / destination.name
        with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:	# CSV와 JSON은 압축하여 용량을 줄인다.
            for name in ("transactions_sample.csv", "articles_sample.csv", "sample_manifest.json"):
                archive.write(work / name, name)
            for article_id in sorted(chosen_images):
                path = image_paths[article_id]
                archive.write(path, f"images/{path.parent.name}/{path.name}", compress_type=ZIP_STORED)	# 이미 압축된 JPEG는 중복 압축 없이 저장한다.
        archive_path.replace(destination)	# 모든 검증과 ZIP 작성을 마친 뒤 최종 파일을 교체한다.
    return destination, manifest	# 완성 경로와 추출 근거를 호출자에게 반환한다.

In [ ]:
dataset_dir = DATASET_DIR if DATASET_DIR is not None else find_dataset(INPUT_ROOT)	# 사용자가 지정한 입력 폴더가 있으면 자동 검색보다 우선한다.
archive, manifest = prepare_hm_sample(	# 설정값을 표본 추출 함수에 전달한다.
    dataset_dir, OUTPUT_DIR,
    start=START_DATE, end=END_DATE, seed=SEED,
    max_customers=MAX_CUSTOMERS, max_images=MAX_IMAGES,
    min_transactions=MIN_TRANSACTIONS, chunk_size=CHUNK_SIZE,
    sales_channel=SALES_CHANNEL,
)
print(f"Archive: {archive.name} ({archive.stat().st_size / (1024 ** 2):.2f} MiB)")	# 완성 ZIP의 이름과 실제 파일 크기를 확인한다.
print(f"Rows: {manifest['transactions']['sampled_rows']:,}")	# 최소 거래 조건에 대응하는 최종 행 수를 표시한다.
print(f"Customers: {manifest['sampled_customers']:,}")
print(f"Product images: {manifest['sampled_images']:,}")
print(f"RFM reference date: {manifest['reference_date']}")	# 후속 RFM 분석에서 사용할 고정 기준일을 표시한다.